# Real-Time Moving Object Detection & Tracking using YOLOv8

## Overview
This notebook demonstrates fine-tuning **YOLOv8** on the real-world **Penn-Fudan Pedestrian Dataset** for moving object detection and real-time Multi-Object Tracking (MOT) using **ByteTrack**.

In [ ]:
import os
import sys
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

# Ensure project src/ directory is importable
module_path = os.path.abspath(os.path.join('..', 'src'))
if module_path not in sys.path:
    sys.path.append(module_path)

from data_loader import load_config, create_data_yaml, download_and_prepare_penn_fudan, explore_dataset_summary
from annotation_converter import xyxy_to_yolo, yolo_to_xyxy, read_yolo_label_file
from model_trainer import load_yolo_model, train_yolo_model
from inference import detect_objects_single_image, calculate_iou, benchmark_inference_speed
from evaluation import evaluate_detection_metrics, generate_detection_report
from visualization import draw_bounding_boxes, plot_precision_recall_curves, plot_confusion_matrix_heatmap, plot_iou_distribution
from tracker import track_moving_objects_video, create_synthetic_moving_pedestrian_video

print(f"Ultralytics YOLO Version Installed. PyTorch Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 1. Load Configuration & Prepare Real-World Pedestrian Dataset

In [ ]:
config = load_config('../config/training_config.yaml')
download_and_prepare_penn_fudan(config)
data_yaml = create_data_yaml(config)
explore_dataset_summary(data_yaml)

## 2. Load Pre-trained YOLOv8 Model & Fine-Tune

In [ ]:
model, results = train_yolo_model(config)

## 3. Single-Image Inference & Visual Overlay

In [ ]:
test_image = '../data/images/test/FudanPed00002.jpg'
if not os.path.exists(test_image):
    test_dir = '../data/images/test'
    test_files = [os.path.join(test_dir, f) for f in os.listdir(test_dir) if f.endswith('.jpg')]
    test_image = test_files[0] if len(test_files) > 0 else 'sample.jpg'

res = detect_objects_single_image(model, test_image, conf_threshold=0.25, iou_threshold=0.45)
img_bgr = cv2.imread(test_image)
annotated = draw_bounding_boxes(img_bgr, res['boxes'], res['classes'], res['scores'], class_names={0: 'person'})

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title('YOLOv8 Real-World Pedestrian Detection', fontsize=14, fontweight='bold')
plt.axis('off')
plt.show()

## 4. Multi-Object Video Tracking (ByteTrack)

In [ ]:
sample_video = '../data/sample_moving_persons.mp4'
if not os.path.exists(sample_video):
    create_synthetic_moving_pedestrian_video(sample_video)

tracking_stats = track_moving_objects_video(
    video_source=sample_video,
    output_video_path='../results/tracked_output.mp4',
    model_path='../models/best.pt',
    tracker='bytetrack.yaml'
)
print("Tracking Finished:", tracking_stats)